# From search-engine output to a protein attention network

This tutorial goes from a **mass-spectrometry search result** (a PSM list deposited on
PRIDE) to a **protein-protein attention network** produced zero-shot by the pretrained
OmicsFM proteomics model - no retraining, no ground-truth labels.

The route:

1. download MaxQuant mzIdentML result files from PRIDE project
   [PXD081816](https://www.ebi.ac.uk/pride/archive/projects/PXD081816)
   (human gastric organoids and tissue, DDA);
2. read them with [`psm_utils`](https://psm-utils.readthedocs.io) - the same code works
   for any format psm_utils supports (mzid, MaxQuant `msms.txt`, Percolator, Sage, ...);
3. turn PSM counts into a per-sample protein abundance profile;
4. run the **ESM-C proteomics checkpoint** on those profiles and accumulate its
   attention into one protein x protein map. Because this checkpoint represents proteins
   by their ESM-C sequence embeddings, it accepts *any* set of UniProt proteins - the
   profile does not have to match the pretraining vocabulary;
5. inspect the strongest pairs, draw the network, and sanity-check the ranking against
   CORUM complexes.

Requirements: the `omicsfm` conda environment plus `pip install psm_utils`.
A GPU helps but is not required for a dataset of this size.

In [1]:
from pathlib import Path
import urllib.request
import numpy as np
import pandas as pd
import anndata as adata_lib

# repo root = parent of tutorials/
ROOT = Path.cwd().resolve()
if ROOT.name == "tutorials":
    ROOT = ROOT.parent
CKPT = ROOT / "model" / "proteomics_esmc" / "best_model.ckpt"
FASTA = ROOT / "fasta" / "human_proteome_canonical_31032022.fasta"
ESMC_CACHE = ROOT / "data" / "esmc_human_cache.pt"
DATA = ROOT / "tutorials" / "data"
DATA.mkdir(exist_ok=True)
assert CKPT.exists() and FASTA.exists(), "run this notebook from the omicsFM checkout"


## 1. Download search-engine output from PRIDE

Two MaxQuant mzIdentML files from PXD081816, each one LC-MS/MS run of a human gastric
sample. Swap in your own files here - anything psm_utils can read works.

In [2]:
BASE = "https://ftp.pride.ebi.ac.uk/pride/data/archive/2026/07/PXD081816/"
RUNS = {
    "organoid_DDA_03": "Experiment_MaxQuant_03_from_CherneM_20260415_01_DDA_03.mzid",
    "gastric_tissue_02": "Experiment_MaxQuant_02_from_BimczokD_022323_02.mzid",
}
for name, fn in RUNS.items():
    dest = DATA / fn
    if not dest.exists():
        print("downloading", fn)
        urllib.request.urlretrieve(BASE + fn, dest)
    print(f"{name} -> {dest.name} ({dest.stat().st_size / 1e6:.1f} MB)")


organoid_DDA_03 -> Experiment_MaxQuant_03_from_CherneM_20260415_01_DDA_03.mzid (18.4 MB)
gastric_tissue_02 -> Experiment_MaxQuant_02_from_BimczokD_022323_02.mzid (17.3 MB)


## 2. Read the PSMs with psm_utils

`read_file` autodetects the format. We drop decoy hits and MaxQuant contaminant /
reversed entries (`CON__`, `REV__`) and count PSMs per UniProt accession. A PSM that
maps to several proteins contributes to each - spectral counting keeps this tutorial
simple; any per-protein abundance vector (MaxQuant `proteinGroups.txt`, a DIA-NN
report, FlashLFQ output, ...) feeds the model the same way.

In [3]:
from psm_utils.io import read_file

def psm_counts(path) -> pd.Series:
    counts = {}
    for psm in read_file(str(path)):
        if psm.is_decoy:
            continue
        for acc in (psm.protein_list or []):
            if acc.startswith(("CON__", "REV__")):
                continue
            counts[acc] = counts.get(acc, 0) + 1
    return pd.Series(counts, name=path.stem)

counts = pd.DataFrame({name: psm_counts(DATA / fn) for name, fn in RUNS.items()}).fillna(0)
print(counts.shape[0], "proteins across", counts.shape[1], "runs")
counts.sort_values(counts.columns[0], ascending=False).head(8)


2190 proteins across 2 runs


,organoid_DDA_03,gastric_tissue_02
P35749,256.0,200.0
P12111,247.0,141.0
P35579,237.0,191.0
P21333,183.0,136.0
P12110,162.0,54.0
Q05707,155.0,100.0
Q01082,132.0,34.0
P98160,124.0,89.0


## 3. PSM counts -> abundance profiles -> AnnData

Longer proteins generate more peptides, so raw spectral counts over-weight them. We
divide each count by the protein's sequence length (SAF, the length-normalised count
underlying NSAF - matching the spectral-count semantics of the pretraining corpus).
OmicsFM bins abundances **per sample by rank**, so any monotone per-protein scale works.

We keep proteins covered by the shipped canonical human FASTA and its precomputed
ESM-C embedding cache: the checkpoint needs a sequence embedding for every protein.

In [4]:
def fasta_lengths_and_genes(fasta_path):
    lengths, genes, acc, seqlen, gene = {}, {}, None, 0, None
    for line in open(fasta_path, encoding="utf8"):
        if line.startswith(">"):
            if acc:
                lengths[acc], genes[acc] = seqlen, gene
            parts = line.split("|")
            acc = parts[1] if len(parts) > 2 else None
            gene = None
            for tok in line.split():
                if tok.startswith("GN="):
                    gene = tok[3:]
            seqlen = 0
        else:
            seqlen += len(line.strip())
    if acc:
        lengths[acc], genes[acc] = seqlen, gene
    return lengths, genes

LEN, GENE = fasta_lengths_and_genes(FASTA)

# the shipped ESM-C cache covers the canonical proteome; drop the handful of
# detected proteins (immunoglobulin variable regions and the like) outside it
import torch
esmc_keys = set(torch.load(ESMC_CACHE, map_location="cpu")["embeddings"].keys())
keep = [p for p in counts.index if p in LEN and p in esmc_keys]
print(f"{len(keep)} of {counts.shape[0]} proteins have a canonical sequence "
      f"and a cached ESM-C embedding")

saf = counts.loc[keep].div(pd.Series({p: LEN[p] for p in keep}), axis=0)

# The proteomics checkpoints collapse proteins with *identical* abundance into
# groups and drop them (a guard against quantification artefacts in the corpus).
# Integer PSM counts produce many exact ties (equal count / equal length), which
# would silently drop hundreds of proteins here. An infinitesimal deterministic
# jitter breaks the ties while leaving each sample's abundance ranking intact.
rng = np.random.default_rng(0)
saf = saf * (1 + 1e-6 * rng.random(saf.shape))

profiles = adata_lib.AnnData(
    X=saf.T.to_numpy(dtype="float32"),
    obs=pd.DataFrame(index=saf.columns),
    var=pd.DataFrame(index=saf.index),
    uns={"modality": "proteomics", "value_semantics": "SAF from PSM spectral counts"},
)
profiles


2170 of 2190 proteins have a canonical sequence and a cached ESM-C embedding


AnnData object with n_obs × n_vars = 2 × 2170
    uns: 'modality', 'value_semantics'
    layers: None (.X)

## 4. Zero-shot attention over the profiles

`attention_map` feeds every sample through the frozen model with **all detected
proteins as context** and averages the transformer's attention over layers, heads and
samples into one symmetric protein x protein map. The model reads at most 1024
proteins per pass, so for a deep sample a single pass covers only a subset of the
protein pairs. `n_epochs` repeats the pass with a freshly sampled 1024-protein window
each time and accumulates a count-weighted average per pair - more passes mean fuller
pair coverage and a stabler attention estimate, so use as many as your hardware
allows (here 100). The ESM-C lookup for our protein set is served from the shipped
`esmc_human_cache.pt`, so no ESM-C inference is needed.

In [5]:
import torch
from omicsfm.attention import attention_map

device = "cuda" if torch.cuda.is_available() else "cpu"
res = attention_map(
    str(CKPT), profiles,
    fasta_path=str(FASTA), esmc_cache=str(ESMC_CACHE),
    device=device, batch_size=2, n_epochs=100,   # lower on CPU-only machines
)
attn, proteins = res["attn"], res["proteins"]
print("attention map:", attn.shape)


Building cache (dense):   0%|          | 0/1 [00:00<?, ?it/s]

Building cache (dense): 100%|██████████| 1/1 [00:00<00:00, 999.36it/s]

attention 1/100:   0%|          | 0/1 [00:00<?, ?it/s]

attention 1/100: 100%|██████████| 1/1 [00:00<00:00,  5.50it/s]

attention 1/100: 100%|██████████| 1/1 [00:00<00:00,  5.45it/s]

...

attention 99/100:   0%|          | 0/1 [00:00<?, ?it/s]

attention 99/100: 100%|██████████| 1/1 [00:00<00:00, 152.29it/s]

attention 100/100:   0%|          | 0/1 [00:00<?, ?it/s]

attention 100/100: 100%|██████████| 1/1 [00:00<00:00, 181.67it/s]

attention map: (2170, 2170)


## 5. The strongest pairs

Rank all protein pairs by mean attention. Known complex partners and pathway
neighbours should populate the top of the list.

In [6]:
iu = np.triu_indices(len(proteins), k=1)
pairs = pd.DataFrame({
    "protein_a": np.array(proteins)[iu[0]],
    "protein_b": np.array(proteins)[iu[1]],
    "attention": attn[iu],
}).dropna().sort_values("attention", ascending=False).reset_index(drop=True)
pairs["gene_a"] = pairs["protein_a"].map(GENE)
pairs["gene_b"] = pairs["protein_b"].map(GENE)
pairs.head(15)


,protein_a,protein_b,attention,gene_a,gene_b
0,P00325,P00326,0.037256,ADH1B,ADH1C
1,P04264,P35908,0.035460,KRT1,KRT2
2,P02533,P08779,0.033928,KRT14,KRT16
3,P02538,P48668,0.032058,KRT6A,KRT6C
4,P13647,P48668,0.030934,KRT5,KRT6C
5,P02538,P13647,0.030289,KRT6A,KRT5
6,P00326,P07327,0.030266,ADH1C,ADH1A
7,P15531,P22392,0.030052,NME1,NME2
8,P00325,P07327,0.029020,ADH1B,ADH1A
9,P61619,Q9H9S3,0.028966,SEC61A1,SEC61A2


## 6. Sanity check against nine reference databases

Are top-attention pairs enriched for known molecular relationships? For each of the
nine reference resources used in the manuscript we compare the positive rate among the
**top 100** scorable attention pairs with the background positive rate over all
scorable pairs (pairs a database does not cover are excluded for that database).
This is the lightweight version of the manuscript's association benchmark;
`omicsfm.attention.eval_attention_heads` runs the full version.

In [7]:
from omicsfm.attention import load_ppi_ground_truth

DBS = ["corum", "bioplex", "huri", "string", "reactome", "kegg", "gocc", "gobp", "omnipath"]
TOP_K = 100
rows = []
for db in DBS:
    gt, gt_proteins = load_ppi_ground_truth(db, gt_dir=str(ROOT / "reference"))
    gt_index = {p: i for i, p in enumerate(gt_proteins)}
    both = pairs["protein_a"].isin(gt_index) & pairs["protein_b"].isin(gt_index)
    ia = pairs.loc[both, "protein_a"].map(gt_index).to_numpy()
    ib = pairs.loc[both, "protein_b"].map(gt_index).to_numpy()
    vals = np.asarray(gt[ia, ib]).ravel()          # 1 / 0 / NaN=unknown, attention-ranked
    vals = vals[~np.isnan(vals)]                   # keep pairs the database covers
    background, top = vals.mean(), vals[:TOP_K].mean()
    rows.append({"database": db, "scorable_pairs": len(vals),
                 "background_rate": background, f"top{TOP_K}_rate": top,
                 "enrichment": top / background})

enrich = pd.DataFrame(rows).set_index("database")
mean_enrichment = enrich["enrichment"].mean()
print(f"mean top-{TOP_K} enrichment across {len(DBS)} databases: {mean_enrichment:.1f}x")
enrich.round(4).sort_values("enrichment", ascending=False)


mean top-100 enrichment across 9 databases: 54.8x


,scorable_pairs,background_rate,top100_rate,enrichment
database,,,,
bioplex,1358658,0.0027,0.47,177.134293
huri,314938,0.0015,0.17,112.477898
string,1743742,0.0116,0.80,69.175499
gobp,1533483,0.0161,0.91,56.437302
corum,368550,0.0187,0.40,21.436701
reactome,1222760,0.0485,0.99,20.408501
omnipath,1524770,0.0006,0.01,16.998600
gocc,1348310,0.0705,0.94,13.341000
kegg,844736,0.1664,0.95,5.709000


## 7. Explore the network interactively

The strongest pairs as a draggable network (pyvis / vis.js): drag nodes around,
hover for the UniProt accession, zoom with the mouse wheel. Node colour = greedy
modularity community, node size = degree, edge width = attention. Isolated
two-protein pairs are dropped so the view concentrates on the actual clusters.

In [8]:
import networkx as nx
from pyvis.network import Network
from IPython.display import IFrame

TOP_EDGES = 250
G = nx.Graph()
for _, r in pairs.head(TOP_EDGES).iterrows():
    a, b = r.gene_a or r.protein_a, r.gene_b or r.protein_b
    G.add_edge(a, b, weight=float(r.attention))
    G.nodes[a]["accession"], G.nodes[b]["accession"] = r.protein_a, r.protein_b

# isolated two-protein pairs clutter the view - keep clusters of 3+ proteins
for comp in list(nx.connected_components(G)):
    if len(comp) < 3:
        G.remove_nodes_from(comp)

communities = list(nx.community.greedy_modularity_communities(G, weight="weight"))
community = {n: i for i, c in enumerate(communities) for n in c}
palette = ["#4c78a8", "#f58518", "#54a24b", "#e45756", "#72b7b2", "#eeca3b",
           "#b279a2", "#ff9da6", "#9d755d", "#bab0ac"]

net = Network(height="750px", width="100%", cdn_resources="remote",
              bgcolor="#ffffff", font_color="#222222")
w_max = max(d["weight"] for _, _, d in G.edges(data=True))
for n in G.nodes:
    net.add_node(n, label=n, title=f"{n} ({G.nodes[n]['accession']}) - degree {G.degree(n)}",
                 color=palette[community[n] % len(palette)], size=12 + 3 * G.degree(n))
for a, b, d in G.edges(data=True):
    net.add_edge(a, b, value=d["weight"] / w_max,
                 title=f"attention {d['weight']:.4f}")
net.barnes_hut(gravity=-4000, spring_length=120)

html_path = DATA / "attention_network.html"
net.save_graph(str(html_path))
print(f"{G.number_of_nodes()} proteins, {G.number_of_edges()} edges, "
      f"{len(communities)} communities")
print(f"saved to {html_path} - open it in a browser if the frame below stays blank")
IFrame(src=str(html_path.relative_to(Path.cwd())).replace(chr(92), "/"), width="100%", height=780)


137 proteins, 165 edges, 35 communities
saved to C:\Projects\external_omicsFM_setup\omicsFM\tutorials\data\attention_network.html - open it in a browser if the frame below stays blank


## Where to go next

- Feed **your own data**: anything that yields a per-protein abundance vector per
  sample becomes an `AnnData` exactly as in step 3.
- `attention_map(..., head=(layer, head))` inspects a single head;
  `symmetrize=False` keeps the directed map.
- The SST tutorial (next in this folder) embeds whole samples with the same
  checkpoints via `omicsfm.api.compute_sst`.